In [56]:
import pandas as pd
import numpy as np
from keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences
from keras.models import Sequential
from keras.layers import Dense, LSTM, Embedding
import random
import re
from tensorflow.keras.utils import to_categorical
import unicodedata

# 🧠 Next Sentence Generator using LSTM – Step-by-Step

> 🔵 **Step 1: Collect a Text Corpus**
- Gather a dataset of sentences or paragraphs.
- Examples: news articles, books, movie dialogues, Wikipedia dumps.

> 🟢 **Step 2: Preprocess the Text**
- Convert text to lowercase.
- Remove punctuation and special characters.
- Tokenize text into words using `Tokenizer`.

> 🟡 **Step 3: Create Sequences**
- For each sentence, create multiple input-output pairs (n-grams).
- Example: "I love deep learning" → ["I love", "I love deep", "I love deep learning"]

> 🟠 **Step 4: Pad the Sequences**
- Use `pad_sequences()` to make all input sequences the same length.
- Padding is usually done from the beginning (`padding='pre'`).

> 🟣 **Step 5: One-Hot Encode the Output**
- Convert target (next word) into one-hot vectors using `to_categorical()`.

> 🔴 **Step 6: Build the LSTM Model**
- Add an `Embedding` layer to convert words to vectors.
- Add an `LSTM` layer (or multiple layers).
- Add a `Dense` output layer with `softmax` activation.

> 🟤 **Step 7: Compile the Model**
- Use `categorical_crossentropy` as the loss function.
- Use `adam` optimizer.
- Track `accuracy` as a metric.

> ⚫ **Step 8: Train the Model**
- Use `model.fit()` with `X` (input sequences) and `y` (output words).
- Train for enough epochs (e.g., 300–500) for good accuracy.

> ⚪ **Step 9: Generate Sentences**
- Start with a seed text (e.g., "I love").
- Predict the next word using the model.
- Append the predicted word to the seed and repeat for desired number of words.



In [57]:
# get corpus file, read it and put it as a list
corpus_file = 'star_wars.txt'
with open(corpus_file, 'r') as star_wars_file:
    corpus = star_wars_file.read().lower().split('\n')
    
print(corpus)

['a long time ago in a galaxy far, far away...', 'i’ve got a bad feeling about this.', 'help me, obi-wan kenobi. you’re my only hope.', 'do or do not. there is no try.', 'the force will be with you. always.', 'i am your father.', 'no. i am your father.', 'the dark side of the force is a pathway to many abilities some consider to be unnatural.', 'fear is the path to the dark side. fear leads to anger. anger leads to hate. hate leads to suffering.', "it's a trap!", 'i find your lack of faith disturbing.', 'i love you. i know.', 'never tell me the odds.', 'this is the way.', 'i am a jedi, like my father before me.', 'chewie, we’re home.', 'the force is strong with this one.', 'your focus determines your reality.', 'you were the chosen one! it was said that you would destroy the sith, not join them!', 'this is where the fun begins.', 'so this is how liberty dies... with thunderous applause.', 'i will not condone a course of action that will lead us to war.', 'in my experience, there is no 

In [58]:
# clean data
def clean_data(text):
    text = unicodedata.normalize('NFKD', text)
    text = text.lower()
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)  # remove non-alphanumeric characters
    text = re.sub(r'\s+', ' ', text) # remove extra spaces
    return text

# clean corpus
clean_corpus = [clean_data(text) for text in corpus]
print(clean_corpus)
    

['a long time ago in a galaxy far far away', 'ive got a bad feeling about this', 'help me obiwan kenobi youre my only hope', 'do or do not there is no try', 'the force will be with you always', 'i am your father', 'no i am your father', 'the dark side of the force is a pathway to many abilities some consider to be unnatural', 'fear is the path to the dark side fear leads to anger anger leads to hate hate leads to suffering', 'its a trap', 'i find your lack of faith disturbing', 'i love you i know', 'never tell me the odds', 'this is the way', 'i am a jedi like my father before me', 'chewie were home', 'the force is strong with this one', 'your focus determines your reality', 'you were the chosen one it was said that you would destroy the sith not join them', 'this is where the fun begins', 'so this is how liberty dies with thunderous applause', 'i will not condone a course of action that will lead us to war', 'in my experience there is no such thing as luck', 'strike me down and i will

In [59]:
tokenizer = Tokenizer(oov_token="<OOV>") # Out of Value token
# fit tokenizer on the cleaned corpus
# This will create a word index mapping each word to a unique integer
# The tokenizer will also handle the OOV token
tokenizer.fit_on_texts(clean_corpus)
"""
Calculates the vocabulary size (number of unique words).
Adds +1 because:
    - The Embedding layer in Keras requires a vocab_size, and 0 is usually reserved (for padding).
    - So if tokenizer.word_index has 100 words, vocab_size = 101.
"""
total_words = len(tokenizer.word_index) + 1
print("Total words:", total_words)

Total words: 162


In [60]:
input_sequences = []
# Create input sequences using the tokenizer
for each_line in clean_corpus:
    token_list = tokenizer.texts_to_sequences([each_line])[0]
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

In [61]:
# pad sequences
max_sequence_len = max([len(x) for x in input_sequences])
input_sequences = pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre')
print("Input sequences shape:", np.array(input_sequences).shape)

Input sequences shape: (268, 23)


In [62]:
X = input_sequences[:, :-1]
y = input_sequences[:, -1]

In [63]:
y = to_categorical(y, num_classes=total_words)

In [64]:
model = Sequential()
model.add(Embedding(total_words, 100, input_length=max_sequence_length-1))
model.add(LSTM(100))
model.add(Dense(total_words, activation='softmax'))

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

Model: "sequential_3"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_3 (Embedding)     (None, 22, 100)           16200     
                                                                 
 lstm_3 (LSTM)               (None, 100)               80400     
                                                                 
 dense_3 (Dense)             (None, 162)               16362     
                                                                 
Total params: 112962 (441.26 KB)
Trainable params: 112962 (441.26 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [65]:
# train the model
model.fit(X, y, epochs=100, verbose=1)


Epoch 1/100
9/9 [==============================] - 1s 9ms/step - loss: 5.0825 - accuracy: 0.0299
Epoch 2/100
9/9 [==============================] - 0s 11ms/step - loss: 5.0231 - accuracy: 0.0560
Epoch 3/100
9/9 [==============================] - 0s 11ms/step - loss: 4.8436 - accuracy: 0.0560
Epoch 4/100
9/9 [==============================] - 0s 13ms/step - loss: 4.7571 - accuracy: 0.0560
Epoch 5/100
9/9 [==============================] - 0s 13ms/step - loss: 4.6998 - accuracy: 0.0373
Epoch 6/100
9/9 [==============================] - 0s 13ms/step - loss: 4.6661 - accuracy: 0.0522
Epoch 7/100
9/9 [==============================] - 0s 12ms/step - loss: 4.6335 - accuracy: 0.0560
Epoch 8/100
9/9 [==============================] - 0s 12ms/step - loss: 4.5901 - accuracy: 0.0746
Epoch 9/100
9/9 [==============================] - 0s 11ms/step - loss: 4.5517 - accuracy: 0.0746
Epoch 10/100
9/9 [==============================] - 0s 11ms/step - loss: 4.4855 - accuracy: 0.0709
Epoch 11/100
9/9 [==

In [66]:
def generate_text(seed_text, next_words, tokenizer, model, max_sequence_len):
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_sequence_len - 1, padding='pre')
        predicted = np.argmax(model.predict(token_list, verbose=0), axis=-1)

        output_word = ""
        for word, index in tokenizer.word_index.items():
            if index == predicted:
                output_word = word
                break
        seed_text += " " + output_word
    return seed_text

In [67]:
# TEST MODEL
seed_text = "luke i am your"
print(generate_text("luke i am your", next_words=2, tokenizer=tokenizer, model=model, max_sequence_len=max_sequence_len))
# This will generate text based on the seed text provided.

luke i am your father father
